In [ ]:
"""
Kaggle notebook — multi-horizon AQI forecasting (single-file version).

Everything from utils/hopsworks_client.py, utils/sequences.py, and
utils/metrics.py is inlined here since Kaggle runs a single script/notebook,
not your project's package structure.

SETUP ON KAGGLE (one-time):
  1. Add-ons -> Secrets -> add a secret named HOPSWORKS_API_KEY with your key
  2. pip installs below run automatically in the first cell

Usage: run top to bottom as-is.
"""

# ── Install deps (Kaggle base image doesn't have hopsworks) ────────────────
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hopsworks"], check=True)

# hopsworks pulls in protobuf<5, but Kaggle's preinstalled tensorflow needs
# protobuf>=5.28. If we `import tensorflow` now it will crash with
# "cannot import name 'runtime_version' from 'google.protobuf'". Force
# protobuf back up BEFORE importing tensorflow (nothing has imported it
# yet, so no kernel restart is needed) — this fixes the tf import without
# touching hopsworks' own functionality, which doesn't care about the
# protobuf version bump.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "protobuf>=5.28,<6"],
    check=True,
)

# hopsworks also pulls cryptography up to a version newer than the
# pyOpenSSL already on Kaggle supports. tensorflow optionally imports
# googleapiclient -> oauth2client -> pyOpenSSL (for a GCE cluster
# resolver we never use), and that import chain breaks with
# "AttributeError: module 'lib' has no attribute 'GEN_EMAIL'" on the old
# pyOpenSSL + new cryptography combo. Upgrade pyOpenSSL to match before
# tensorflow gets imported.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pyopenssl"],
    check=True,
)

import logging
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow import keras
from tensorflow.keras import layers

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("aqi_training")

# ── Constants ─────────────────────────────────────────────────────────────
FEATURE_GROUP_NAME = "aqi_features"
FEATURE_GROUP_VERSION = 1

# --- ADJUST to match your actual engineered column names -------------------
TARGET_COLUMNS = ["target_aqi_24h", "target_aqi_48h", "target_aqi_72h"]
HORIZONS = [24, 48, 72]
# -----------------------------------------------------------------------

LOOKBACK_HOURS = 72
GAP_INTERPOLATE_LIMIT = 2
HOLDOUT_FRAC = 0.15
VAL_FRAC = 0.10
BATCH_SIZE = 32
MAX_EPOCHS = 100
PATIENCE = 10
LOSS_WEIGHTS = [1.0, 1.2, 1.5]
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)


# ── Hopsworks connection ─────────────────────────────────────────────────
def get_feature_store():
    import hopsworks

    # Prefer Kaggle Secrets; fall back to env var if you set one manually.
    api_key = None
    try:
        from kaggle_secrets import UserSecretsClient
        api_key = UserSecretsClient().get_secret("HOPSWORKS_API_KEY")
    except Exception:
        api_key = os.environ.get("HOPSWORKS_API_KEY")

    if not api_key:
        raise RuntimeError(
            "No Hopsworks API key found. Add it under Add-ons -> Secrets as "
            "HOPSWORKS_API_KEY, or set the HOPSWORKS_API_KEY environment variable."
        )

    project = hopsworks.login(api_key_value=api_key)
    return project.get_feature_store()


def load_feature_group() -> pd.DataFrame:
    logger.info("Connecting to Hopsworks and reading '%s' v%d", FEATURE_GROUP_NAME, FEATURE_GROUP_VERSION)
    fs = get_feature_store()
    fg = fs.get_feature_group(FEATURE_GROUP_NAME, version=FEATURE_GROUP_VERSION)
    df = fg.read()
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df.sort_values("timestamp").reset_index(drop=True)
    logger.info("Loaded %d rows, %d columns from Hopsworks", *df.shape)
    return df


# ── Gap handling ──────────────────────────────────────────────────────────
def reindex_to_hourly_grid(df: pd.DataFrame, interpolate_limit: int = 2) -> pd.DataFrame:
    """
    Force onto a complete hourly index so hours missing from Hopsworks
    (failed pipeline runs) become explicit NaN rows instead of silently
    vanishing. Short gaps get interpolated; longer gaps stay NaN and get
    excluded from sequence-building below.
    """
    df = df.sort_values("timestamp").reset_index(drop=True)
    full_range = pd.date_range(df["timestamp"].min(), df["timestamp"].max(), freq="h", tz=df["timestamp"].dt.tz)
    df = df.set_index("timestamp").reindex(full_range).rename_axis("timestamp").reset_index()

    n_missing = df.drop(columns="timestamp").isna().any(axis=1).sum()
    logger.info("Reindexed to full hourly grid: %d rows, %d had at least one missing value", len(df), n_missing)

    non_ts_cols = [c for c in df.columns if c != "timestamp"]
    df[non_ts_cols] = df[non_ts_cols].interpolate(method="linear", limit=interpolate_limit, limit_area="inside")

    still_missing = df.drop(columns="timestamp").isna().any(axis=1).sum()
    if still_missing:
        logger.warning(
            "%d rows still NaN after interpolation (gaps longer than %d hours) — "
            "any window touching these will be dropped, not fabricated.",
            still_missing, interpolate_limit,
        )
    return df


def get_feature_cols(df: pd.DataFrame, target_cols: list[str]) -> list[str]:
    exclude = {"timestamp", "has_target"} | set(target_cols)
    return [c for c in df.columns if c not in exclude]


# ── Sequence building (gap-aware) ────────────────────────────────────────
def build_sequences(df, feature_cols, target_cols, lookback=LOOKBACK_HOURS):
    work = df.copy()
    if not work["timestamp"].is_monotonic_increasing:
        raise ValueError("df must be sorted by timestamp ascending")

    target_mask = work[target_cols].notna().all(axis=1)
    valid_idx = work.index[target_mask].to_list()

    feature_array = work[feature_cols].to_numpy(dtype=np.float32)
    target_array = work[target_cols].to_numpy(dtype=np.float32)
    timestamp_array = work["timestamp"].to_numpy()

    samples_X, samples_y, samples_ts = [], [], []
    dropped_gap, dropped_nan, dropped_short = 0, 0, 0

    for i in valid_idx:
        if i - lookback + 1 < 0:
            dropped_short += 1
            continue

        ts_window = timestamp_array[i - lookback + 1: i + 1]
        deltas = np.diff(ts_window).astype("timedelta64[h]")
        if not np.all(deltas == np.timedelta64(1, "h")):
            dropped_gap += 1
            continue

        window = feature_array[i - lookback + 1: i + 1]
        if np.isnan(window).any():
            dropped_nan += 1
            continue

        samples_X.append(window)
        samples_y.append(target_array[i])
        samples_ts.append(timestamp_array[i])

    logger.info(
        "Window filtering: %d kept, %d dropped (gap-spanning), %d dropped (NaN feature), %d dropped (short history)",
        len(samples_X), dropped_gap, dropped_nan, dropped_short,
    )

    if not samples_X:
        raise ValueError("No valid sequences could be built. Check lookback, NaN handling, gap frequency, and target alignment.")

    X = np.stack(samples_X, axis=0)
    y = np.stack(samples_y, axis=0)
    timestamps = pd.DatetimeIndex(samples_ts)
    logger.info("Built %d sequences | X %s | y %s | %s -> %s", len(X), X.shape, y.shape, timestamps[0], timestamps[-1])
    return X, y, timestamps


# ── Split / scale ─────────────────────────────────────────────────────────
def chronological_split(X, y, timestamps):
    n = len(X)
    holdout_start = int(n * (1 - HOLDOUT_FRAC))
    val_start = int(holdout_start * (1 - VAL_FRAC))

    X_train, y_train = X[:val_start], y[:val_start]
    X_val, y_val = X[val_start:holdout_start], y[val_start:holdout_start]
    X_test, y_test = X[holdout_start:], y[holdout_start:]
    ts_test = timestamps[holdout_start:]

    logger.info("Split — train: %d  val: %d  holdout: %d", len(X_train), len(X_val), len(X_test))
    return X_train, y_train, X_val, y_val, X_test, y_test, ts_test


def scale(X_train, X_val, X_test):
    n_train, t, f = X_train.shape
    scaler = StandardScaler()
    X_train_2d = scaler.fit_transform(X_train.reshape(-1, f))
    X_val_2d = scaler.transform(X_val.reshape(-1, f))
    X_test_2d = scaler.transform(X_test.reshape(-1, f))
    return (
        X_train_2d.reshape(n_train, t, f),
        X_val_2d.reshape(len(X_val), t, f),
        X_test_2d.reshape(len(X_test), t, f),
        scaler,
    )


# ── Model ─────────────────────────────────────────────────────────────────
def build_model(lookback, n_features):
    inp = keras.Input(shape=(lookback, n_features), name="sequence_input")
    x = layers.LSTM(64, return_sequences=True, name="lstm_1")(inp)
    x = layers.Dropout(0.2, name="drop_1")(x)
    x = layers.LSTM(32, return_sequences=False, name="lstm_2")(x)
    x = layers.Dropout(0.2, name="drop_2")(x)
    x = layers.Dense(16, activation="relu", name="shared_dense")(x)

    out_24h = layers.Dense(1, name="out_24h")(x)
    out_48h = layers.Dense(1, name="out_48h")(x)
    out_72h = layers.Dense(1, name="out_72h")(x)

    model = keras.Model(inputs=inp, outputs=[out_24h, out_48h, out_72h])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss={"out_24h": "mse", "out_48h": "mse", "out_72h": "mse"},
        loss_weights={"out_24h": LOSS_WEIGHTS[0], "out_48h": LOSS_WEIGHTS[1], "out_72h": LOSS_WEIGHTS[2]},
        metrics={"out_24h": "mae", "out_48h": "mae", "out_72h": "mae"},
    )
    return model


def train(model, X_train, y_train, X_val, y_val):
    early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1)
    return model.fit(
        X_train,
        [y_train[:, 0], y_train[:, 1], y_train[:, 2]],
        validation_data=(X_val, [y_val[:, 0], y_val[:, 1], y_val[:, 2]]),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=1,
    )


def regression_metrics(y_true, y_pred) -> dict:
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    if len(y_true) == 0:
        return {"rmse": float("nan"), "mae": float("nan"), "r2": float("nan")}
    return {
        "rmse": round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 3),
        "mae": round(float(mean_absolute_error(y_true, y_pred)), 3),
        "r2": round(float(r2_score(y_true, y_pred)), 3),
    }


def evaluate_model(model, X_test, y_test) -> dict:
    preds = model.predict(X_test, verbose=0)
    return {h: regression_metrics(y_test[:, i], preds[i].squeeze()) for i, h in enumerate(HORIZONS)}


# ── Run everything ───────────────────────────────────────────────────────
df = load_feature_group()

missing = [c for c in TARGET_COLUMNS if c not in df.columns]
if missing:
    raise ValueError(
        f"Expected target columns not found: {missing}. "
        f"Update TARGET_COLUMNS at the top of this script. "
        f"Available columns: {list(df.columns)}"
    )

df = reindex_to_hourly_grid(df, interpolate_limit=GAP_INTERPOLATE_LIMIT)
feature_cols = get_feature_cols(df, TARGET_COLUMNS)
logger.info("Using %d feature columns, %d targets", len(feature_cols), len(TARGET_COLUMNS))

X, y, timestamps = build_sequences(df, feature_cols, TARGET_COLUMNS, lookback=LOOKBACK_HOURS)
X_train, y_train, X_val, y_val, X_test, y_test, ts_test = chronological_split(X, y, timestamps)
X_train, X_val, X_test, scaler = scale(X_train, X_val, X_test)

model = build_model(lookback=LOOKBACK_HOURS, n_features=X_train.shape[2])
model.summary()

history = train(model, X_train, y_train, X_val, y_val)
epochs_run = len(history.history["loss"])
logger.info("Stopped at epoch %d / %d", epochs_run, MAX_EPOCHS)

plt.plot(history.history["loss"], label="train_loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.legend()
plt.title("Training loss")
plt.savefig("/kaggle/working/training_loss.png")
plt.show()

results = evaluate_model(model, X_test, y_test)

print("\n" + "=" * 60)
print("LSTM RESULTS (holdout, never seen during training or early stopping)")
print("=" * 60)
for h in HORIZONS:
    r = results[h]
    print(f"  {h}h  RMSE={r['rmse']:.2f}  MAE={r['mae']:.2f}  R²={r['r2']:.3f}")
print("=" * 60)

model.save("/kaggle/working/aqi_multi_horizon_lstm.keras")
logger.info("Model saved to /kaggle/working/aqi_multi_horizon_lstm.keras")